# Análisis de Riesgo de Mora en Clientes — XGBoost y SHAP
## Limpieza, Preparación y Modelamiento de Datos

Este notebook implementa un análisis completo de clasificación de riesgo utilizando técnicas de machine learning con interpretabilidad basada en SHAP.

## PASO 1: Importar Librerías y Cargar Base de Datos

### Descripción
Se importan las librerías necesarias para el análisis de datos y se carga la base de datos de financiación de clientes.

In [1]:
import pandas as pd
import numpy as np

In [3]:
df_financiacion = pd.read_excel('/Users/yedisoncuervo/Desktop/Proyecto-4_Analitica_III/Datos/BD taller clasificación (3).xlsx')
df_financiacion.head(5)

,Caso,Perfil,Estado,Edad,Genero,ScoreCrediticio,PorcentajeFinanciacion,Plazo,IngresoEstimado,Gastos,CapacidadDePago,ValorCuotaMensual,M3_30AC
0,1004991730,ASALARIADO,NUEVO,30,FEMENINO,748,0.6850,72,3289800.0,2430508.51,0.361093,2379693,0
1,1005097331,INDEPENDIENTE,NUEVO,46,MASCULINO,670,0.2783,60,2425440.0,1621788.08,0.948770,847046,0
2,1005120587,INDEPENDIENTE,USADO,39,MASCULINO,752,1.0000,60,30000000.0,3614018.63,12.009213,2197145,0
3,1005152562,ASALARIADO,USADO,38,FEMENINO,758,1.0000,84,1631331.0,1725244.99,-0.068706,1366896,0
4,1005153782,INDEPENDIENTE,NUEVO,60,FEMENINO,846,0.6596,72,20907400.0,3439341.88,13.004595,1343222,0


## PASO 2: Información General de la Base de Datos

### Descripción
Se obtiene información general del dataset: estructura, tipos de datos, cantidad de registros y columnas.

In [4]:
df_financiacion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21091 entries, 0 to 21090
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Caso                    21091 non-null  int64  
 1   Perfil                  21091 non-null  object 
 2   Estado                  21091 non-null  object 
 3   Edad                    21091 non-null  int64  
 4   Genero                  21091 non-null  object 
 5   ScoreCrediticio         21091 non-null  int64  
 6   PorcentajeFinanciacion  21091 non-null  float64
 7   Plazo                   21091 non-null  int64  
 8   IngresoEstimado         21063 non-null  float64
 9   Gastos                  21091 non-null  float64
 10  CapacidadDePago         21063 non-null  float64
 11  ValorCuotaMensual       21091 non-null  int64  
 12  M3_30AC                 21091 non-null  int64  
dtypes: float64(4), int64(6), object(3)
memory usage: 2.1+ MB


## PASO 3: Validación de Datos Faltantes, Nulos y Duplicados

### Descripción
Se realiza un análisis exhaustivo de la calidad de datos:
- **Datos nulos**: Se identifican y cuentan valores faltantes por variable
- **Datos duplicados**: Se detectan registros duplicados
- **Criterio de eliminación**: Si una columna tiene más del 40% de datos faltantes, será eliminada
- **Manejo de nulos**: Se eliminarán filas con valores faltantes si el porcentaje es bajo

In [31]:
# ── 4. Datos nulos ────────────────────────────────────────────────────────
print("DATOS NULOS")
nulos = df_financiacion.isnull().sum()
pct   = (nulos / len(df_financiacion) * 100).round(2)
resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': pct})
resumen_nulos = resumen_nulos[resumen_nulos['Nulos'] > 0]
if resumen_nulos.empty:
    print("No hay datos nulos.")
else:
    print(resumen_nulos)

DATOS NULOS
                 Nulos  Porcentaje (%)
IngresoEstimado     28            0.13
CapacidadDePago     28            0.13


### Tratamiento de Datos Nulos
Dado que el porcentaje de datos nulos en las variables (IngresosEstimado/CapacidadDePago) es de solo **0.13%**, se procede a **eliminar estas filas** de la base de datos, manteniendo la integridad de los datos.

In [32]:
# Eliminar filas con datos nulos
df_financiacion = df_financiacion.dropna()

In [33]:
# Verificar que quedaron cero nulos
print(f"Filas después de eliminar nulos: {len(df_financiacion)}")
print(f"Nulos restantes: {df_financiacion.isnull().sum().sum()}")

Filas después de eliminar nulos: 21063
Nulos restantes: 0


In [34]:
# ── 5. Datos duplicados ───────────────────────────────────────────────────
print("DATOS DUPLICADOS")
duplicados = df_financiacion.duplicated().sum()
print(f"Filas duplicadas: {duplicados}")

DATOS DUPLICADOS
Filas duplicadas: 0


In [35]:
# ── 6. Estadísticas descriptivas ──────────────────────────────────────────
print("ESTADÍSTICAS DESCRIPTIVAS")
print(df_financiacion.describe())

ESTADÍSTICAS DESCRIPTIVAS
               Caso          Edad  ScoreCrediticio  PorcentajeFinanciacion  \
count  2.106300e+04  21063.000000     21063.000000            21063.000000   
mean   1.006178e+09     44.542563       782.361724                0.743488   
std    3.264530e+05     12.744980        85.314167                0.246626   
min    1.004992e+09     19.000000       343.000000                0.100000   
25%    1.005912e+09     34.000000       726.000000                0.552200   
50%    1.006159e+09     43.000000       783.000000                0.800000   
75%    1.006453e+09     54.000000       838.000000                1.000000   
max    1.006786e+09     75.000000       999.000000                1.067000   

              Plazo  IngresoEstimado        Gastos  CapacidadDePago  \
count  21063.000000     2.106300e+04  2.106300e+04     2.106300e+04   
mean      60.633101     5.018901e+06  1.142708e+08    -8.077237e+01   
std       12.497081     5.955286e+06  1.624658e+10     1.1

### Análisis de Desbalance en la Variable Objetivo
**IMPORTANTE**: Este análisis identifica si existe desbalance severo en la variable objetivo `M3_30AC` (Riesgo de Mora).
Un desbalance severo afectará el rendimiento del modelo y requerirá técnicas de balanceo. 

In [36]:
# ── 7. Distribución de la variable objetivo ───────────────────────────────
print("DISTRIBUCIÓN VARIABLE OBJETIVO (M3_30AC)")
conteo = df_financiacion['M3_30AC'].value_counts()
pct_target = (conteo / len(df_financiacion) * 100).round(2)
print(pd.DataFrame({'Conteo': conteo, 'Porcentaje (%)': pct_target}))

DISTRIBUCIÓN VARIABLE OBJETIVO (M3_30AC)
         Conteo  Porcentaje (%)
M3_30AC                        
0         20228           96.04
1           835            3.96


## PASO 4: Función de Muestreo Estratificado y Codificación de Variables

### Descripción
Se define una función que genera **muestras aleatorias diferentes** del dataset original (sin balancear), manteniendo la proporción de clases. El balanceo se aplicará a cada muestra en la Sección 2.

**Ventajas de este enfoque:**
- Cada muestra selecciona clientes DIFERENTES de la clase minoritaria
- Valida robustez del modelo con múltiples corridas
- Mayor variabilidad y representatividad
- Cambiar `random_state` genera nuevas muestras diferentes

In [ ]:
def generar_muestra_estratificada(df, proporcion=1.0, random_state=42):
    """
    Genera una muestra aleatoria estratificada preservando la proporción de clases.
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame original (con o sin balance)
    proporcion : float (default=1.0)
        Proporción de datos a muestrear (0.0 a 1.0)
        - 0.8 = 80% de los datos
        - 1.0 = 100% de los datos
    random_state : int (default=42)
        Semilla aleatoria para reproducibilidad
        Cambiar este valor genera muestras diferentes
    
    Retorna:
    --------
    pd.DataFrame : Muestra estratificada del dataset
    
    Nota:
    -----
    Esta función se usa con df_financiacion (original).
    El balanceo se aplica DESPUÉS a cada muestra en la Sección 2.
    """
    from sklearn.model_selection import train_test_split
    
    if not (0 < proporcion <= 1.0):
        raise ValueError("La proporción debe estar entre 0 y 1")
    
    # Si proporción = 1.0, retornar la muestra completa
    if proporcion == 1.0:
        return df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    
    # Muestreo estratificado por clase
    muestra, _ = train_test_split(
        df,
        test_size=(1 - proporcion),
        stratify=df['M3_30AC'],
        random_state=random_state
    )
    
    return muestra.reset_index(drop=True)


# ── Demostración de la función ────────────────────────────────────────────
print("FUNCIÓN DE MUESTREO ESTRATIFICADO")
print("=" * 70)
print(f"\n📊 Dataset original (SIN balancear):")
print(f"   Total: {len(df_financiacion)} registros")
print(f"   Sin mora (0): {(df_financiacion['M3_30AC'] == 0).sum()}")
print(f"   Con mora (1): {(df_financiacion['M3_30AC'] == 1).sum()}")
print(f"   Proporción: {(df_financiacion['M3_30AC'] == 0).sum() / (df_financiacion['M3_30AC'] == 1).sum():.1f}:1 (DESBALANCEADO)")

# Generar muestra de ejemplo
muestra_demo = generar_muestra_estratificada(df_financiacion, proporcion=1.0, random_state=42)
print(f"\n📊 Muestra estratificada (100%, random_state=42):")
print(f"   Total: {len(muestra_demo)} registros")
print(f"   Sin mora (0): {(muestra_demo['M3_30AC'] == 0).sum()}")
print(f"   Con mora (1): {(muestra_demo['M3_30AC'] == 1).sum()}")
print(f"   Proporción: {(muestra_demo['M3_30AC'] == 0).sum() / (muestra_demo['M3_30AC'] == 1).sum():.1f}:1 (MANTIENE DESBALANCE)")

print("\n✅ Función lista para usar en la Sección 2")
print("=" * 70)

Total registros: 2505

Distribución variable objetivo:
         Conteo  Porcentaje (%)
M3_30AC                        
0          1670           66.67
1           835           33.33


## PASO 5: Codificación de Variables Categóricas

### Descripción
Se convierten las variables categóricas binarias en formato numérico. Esta codificación se aplicará al dataset original antes de cualquier muestreo.

**Variables a codificar:**
- `Genero`: MASCULINO = 1, FEMENINO = 0
- `Perfil`: ASALARIADO = 1, INDEPENDIENTE = 0  
- `Estado`: NUEVO = 1, USADO = 0

In [ ]:
# Verificar los valores únicos de variables categóricas antes de codificar
print("VALORES ÚNICOS EN VARIABLES CATEGÓRICAS")
print("=" * 50)
print(f"Genero: {df_financiacion['Genero'].unique()}")
print(f"Perfil: {df_financiacion['Perfil'].unique()}")
print(f"Estado: {df_financiacion['Estado'].unique()}")
print("=" * 50)

Estado
NUEVO    16080
USADO     4983
Name: count, dtype: int64
Perfil
ASALARIADO       11588
INDEPENDIENTE     9475
Name: count, dtype: int64
Genero
MASCULINO    11382
FEMENINO      9681
Name: count, dtype: int64


### Aplicar Codificación al Dataset Original

In [ ]:
# ── Codificación de variables categóricas (binarias) ──────────────────────
df_financiacion['Genero']  = df_financiacion['Genero'].map({'MASCULINO': 1, 'FEMENINO': 0})
df_financiacion['Perfil']  = df_financiacion['Perfil'].map({'ASALARIADO': 1, 'INDEPENDIENTE': 0})
df_financiacion['Estado']  = df_financiacion['Estado'].map({'NUEVO': 1, 'USADO': 0})

print(" Codificación aplicada correctamente")
print(f"Verificación - Valores únicos después de codificar:")
print(f"Genero: {df_financiacion['Genero'].unique()}")
print(f"Perfil: {df_financiacion['Perfil'].unique()}")
print(f"Estado: {df_financiacion['Estado'].unique()}")

Genero  Perfil  Estado
1       1       1         587
        0       1         438
0       1       1         432
        0       1         359
1       1       0         229
        0       0         162
0       1       0         150
        0       0         148
Name: count, dtype: int64


In [ ]:
# Visualizar primeras filas del dataset codificado
print("Primeras filas del dataset después de codificar variables:")
df_financiacion.head()

,Caso,Perfil,Estado,Edad,Genero,ScoreCrediticio,PorcentajeFinanciacion,Plazo,IngresoEstimado,Gastos,CapacidadDePago,ValorCuotaMensual,M3_30AC
0,1006446346,1,1,37,1,795,0.9170,60,2388592.0,1573260.21,0.547853,1488231,0
1,1005917646,1,1,35,1,785,1.0112,72,2814350.0,3445278.85,-0.258608,2439711,1
2,1006623892,1,0,34,0,785,1.0000,60,1468462.0,1299911.13,0.108899,1547771,0
3,1006109939,1,1,46,0,816,1.0268,60,1753400.0,1631414.91,0.083970,1452719,1
4,1005989107,1,1,71,0,818,1.0000,72,28641580.0,10610617.88,10.854767,1661110,0


In [ ]:
# ── Eliminar variables que no aportan información ─────────────────────────
# Se elimina la columna 'Caso' que es solo un identificador de fila
df_financiacion = df_financiacion.drop(columns=['Caso'])

print("✅ Dataset preparado para muestreo y modelamiento")
print(f"\n📊 Resumen final:")
print(f"   Total de registros: {len(df_financiacion)}")
print(f"   Total de variables: {df_financiacion.shape[1]}")
print(f"   Sin mora (Clase 0): {(df_financiacion['M3_30AC'] == 0).sum()}")
print(f"   Con mora (Clase 1): {(df_financiacion['M3_30AC'] == 1).sum()}")
print(f"   Proporción original: {(df_financiacion['M3_30AC'] == 0).sum() / (df_financiacion['M3_30AC'] == 1).sum():.1f}:1 (DESBALANCEADO)")
print(f"\n   ⚠️ El balanceo se aplicará a cada muestra en la Sección 2")

,Perfil,Estado,Edad,Genero,ScoreCrediticio,PorcentajeFinanciacion,Plazo,IngresoEstimado,Gastos,CapacidadDePago,ValorCuotaMensual,M3_30AC
0,1,1,37,1,795,0.9170,60,2388592.0,1573260.21,0.547853,1488231,0
1,1,1,35,1,785,1.0112,72,2814350.0,3445278.85,-0.258608,2439711,1
2,1,0,34,0,785,1.0000,60,1468462.0,1299911.13,0.108899,1547771,0
3,1,1,46,0,816,1.0268,60,1753400.0,1631414.91,0.083970,1452719,1
4,1,1,71,0,818,1.0000,72,28641580.0,10610617.88,10.854767,1661110,0


---

## Resumen de la Preparación de Datos

### Transformaciones Realizadas:
 **Datos nulos**: Eliminadas filas con valores faltantes (0.13%)  
 **Datos duplicados**: Verificación completada  
 **Codificación**: Convertidas variables categóricas a numéricas  
 **Limpieza**: Eliminada columna ID innecesaria  
 **Función de muestreo**: Definida para generar muestras estratificadas  

### Dataset Preparado (SIN BALANCEAR AÚN):
- **Total de registros**: 21,330 observaciones
- **Total de variables**: 10 features + 1 target
- **Variable objetivo**: M3_30AC (Sin mora=0, Con mora=1)
- **Desbalance**: Proporción 24:1 (20,493 sin mora, 837 con mora)

### Estrategia para la Sección 2:
1. **Generar múltiples muestras** con `generar_muestra_estratificada()`
   - Cambiar `random_state` para cada corrida
   - Mantiene proporción de clases en cada muestra

2. **Aplicar balanceo a cada muestra** con undersampling 1:2
   - 837 casos con mora (clase 1)
   - 1,674 casos sin mora (clase 0)
   - Resultado: 2,511 registros balanceados por muestra

3. **Entrenar modelos** con cada muestra balanceada
   - Regresión Logística (con escalamiento)
   - XGBoost (sin escalamiento)
   - Validar robustez con múltiples corridas

---

# SECCIÓN 2: MODELAMIENTO Y CLASIFICACIÓN

En esta sección se desarrollarán, optimizarán e interpretarán modelos de machine learning para predecir el riesgo de mora en clientes.

**Flujo de trabajo:**
1. Generar muestra estratificada con `random_state`
2. Aplicar balanceo undersampling 1:2
3. Entrenar Regresión Logística y XGBoost
4. Comparar desempeño de modelos
5. Análisis SHAP para interpretabilidad
6. Validación manual con 10 clientes ficticios

## Importar Librerías para Modelamiento

### Descripción
Se importan las librerías necesarias de scikit-learn, XGBoost y utilidades para modelamiento.

In [ ]:
# ── Librerías para modelamiento ──────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, classification_report, roc_curve, auc)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas correctamente")
print(f"   - Scikit-learn")
print(f"   - XGBoost")
print(f"   - Matplotlib/Seaborn")

---

## Función para Aplicar Balanceo a Cada Muestra

### Descripción
Se define una función que aplica balanceo mediante undersampling 1:2 a una muestra dada. Esta función se usará en cada corrida con diferentes muestras.

In [ ]:
def aplicar_balanceo(df, ratio=2, random_state=42):
    """
    Aplica balanceo mediante undersampling con ratio clase minoritaria:clase mayoritaria.
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame a balancear
    ratio : int (default=2)
        Ratio de balanceo: clase_mayoritaria / clase_minoritaria
        - ratio=2 → 1:2 (1 caso con mora, 2 sin mora)
    random_state : int (default=42)
        Semilla aleatoria para reproducibilidad
    
    Retorna:
    --------
    pd.DataFrame : DataFrame balanceado
    """
    
    clase_1 = df[df['M3_30AC'] == 1]
    clase_0 = df[df['M3_30AC'] == 0]
    
    # Muestrear clase mayoritaria (0) según ratio
    n_clase_0 = len(clase_1) * ratio
    clase_0_muestra = clase_0.sample(n=min(n_clase_0, len(clase_0)), random_state=random_state)
    
    # Combinar y mezclar
    df_balanceado = pd.concat([clase_1, clase_0_muestra]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    return df_balanceado


# ── Demostración de la función ────────────────────────────────────────────
print("FUNCIÓN DE BALANCEO (Undersampling 1:2)")
print("=" * 70)

# Generar una muestra de ejemplo
muestra_demo = generar_muestra_estratificada(df_financiacion, random_state=42)
print(f"\nAntes de balanceo:")
print(f"   Total: {len(muestra_demo)}")
print(f"   Sin mora (0): {(muestra_demo['M3_30AC'] == 0).sum()}")
print(f"   Con mora (1): {(muestra_demo['M3_30AC'] == 1).sum()}")
print(f"   Proporción: {(muestra_demo['M3_30AC'] == 0).sum() / (muestra_demo['M3_30AC'] == 1).sum():.1f}:1")

# Aplicar balanceo
muestra_balanceada = aplicar_balanceo(muestra_demo, ratio=2, random_state=42)
print(f"\nDespués de balanceo (ratio 1:2):")
print(f"   Total: {len(muestra_balanceada)}")
print(f"   Sin mora (0): {(muestra_balanceada['M3_30AC'] == 0).sum()}")
print(f"   Con mora (1): {(muestra_balanceada['M3_30AC'] == 1).sum()}")
print(f"   Proporción: {(muestra_balanceada['M3_30AC'] == 0).sum() / (muestra_balanceada['M3_30AC'] == 1).sum():.1f}:1")

print("\n✅ Función lista para usar en cada corrida")
print("=" * 70)

---

## CORRIDA 1: Configuración y División Train/Test

### Descripción
Se configura la primera corrida con `random_state=42`, se genera la muestra, se aplica balanceo y se divide en conjuntos de entrenamiento y prueba.

**Parámetros de la corrida:**
- `random_state=42`: Semilla para reproducibilidad
- `test_size=0.2`: 80% entrenamiento, 20% prueba
- `stratify`: División estratificada por clase

In [ ]:
# ── CORRIDA 1: Generar muestra y aplicar balanceo ──────────────────────
RANDOM_STATE_CORRIDA = 42

# Generar muestra estratificada
df_muestra = generar_muestra_estratificada(df_financiacion, proporcion=1.0, random_state=RANDOM_STATE_CORRIDA)

# Aplicar balanceo
df_datos = aplicar_balanceo(df_muestra, ratio=2, random_state=RANDOM_STATE_CORRIDA)

print("CONFIGURACIÓN DE LA CORRIDA 1")
print("=" * 70)
print(f"\n📊 Dataset balanceado:")
print(f"   Total de registros: {len(df_datos)}")
print(f"   Sin mora (0): {(df_datos['M3_30AC'] == 0).sum()}")
print(f"   Con mora (1): {(df_datos['M3_30AC'] == 1).sum()}")
print(f"   Proporción: 1:{(df_datos['M3_30AC'] == 0).sum() / (df_datos['M3_30AC'] == 1).sum():.2f}")
print(f"   random_state: {RANDOM_STATE_CORRIDA}")

# ── Preparar features y target ────────────────────────────────────────────
# Eliminar columnas no numéricas o innecesarias
X = df_datos.drop(columns=['M3_30AC'])
y = df_datos['M3_30AC']

print(f"\n📊 Variables de entrada (features):")
print(f"   Total de features: {X.shape[1]}")
print(f"   Nombres: {list(X.columns)}")

# ── División Train/Test ───────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE_CORRIDA
)

print(f"\n📊 División Train/Test (80/20 estratificada):")
print(f"   Entrenamiento: {len(X_train)} registros ({len(X_train)/len(df_datos)*100:.1f}%)")
print(f"      - Sin mora: {(y_train == 0).sum()}")
print(f"      - Con mora: {(y_train == 1).sum()}")
print(f"\n   Prueba: {len(X_test)} registros ({len(X_test)/len(df_datos)*100:.1f}%)")
print(f"      - Sin mora: {(y_test == 0).sum()}")
print(f"      - Con mora: {(y_test == 1).sum()}")

print("\n✅ Datos preparados para entrenamiento de modelos")
print("=" * 70)

---

## MODELO 1: Regresión Logística (CON Escalamiento)

### Descripción
Se entrena un modelo de Regresión Logística con escalamiento StandardScaler. La regresión logística es un modelo lineal que asume relaciones lineales entre features y probabilidad de riesgo.

In [ ]:
# Entrenar Regresión Logística (BASE)
print("MODELO 1: REGRESIÓN LOGÍSTICA (BASE)")
print("=" * 70)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE_CORRIDA)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
y_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
roc_lr = roc_auc_score(y_test, y_proba_lr)

print(f"Accuracy:  {acc_lr:.4f}")
print(f"Precisión: {prec_lr:.4f}")
print(f"Recall:    {rec_lr:.4f}")
print(f"F1-Score:  {f1_lr:.4f}")
print(f"ROC-AUC:   {roc_lr:.4f}")
print("=" * 70)

---

## MODELO  2: XGBoost (Base) 

### Descripción
Se entrena un modelo XGBoost (eXtreme Gradient Boosting) 
XGBoost es un modelo basado en árboles que puede capturar relaciones no lineales y es muy potente.

In [ ]:
# Entrenar XGBoost (BASE)
print("\nMODELO 2: XGBOOST (BASE)")
print("=" * 70)

xgb_model = XGBClassifier(random_state=RANDOM_STATE_CORRIDA, eval_metric='logloss', verbose=0)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

acc_xgb = accuracy_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb)
rec_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)
roc_xgb = roc_auc_score(y_test, y_proba_xgb)

print(f"Accuracy:  {acc_xgb:.4f}")
print(f"Precisión: {prec_xgb:.4f}")
print(f"Recall:    {rec_xgb:.4f}")
print(f"F1-Score:  {f1_xgb:.4f}")
print(f"ROC-AUC:   {roc_xgb:.4f}")
print("=" * 70)

---

## Comparación de Modelos BASE

Tabla comparativa de los dos modelos en su forma base.


In [ ]:
print("\nCOMPARACIÓN - MODELOS BASE")
print("=" * 80)

# Crear tabla comparativa
comparativa_base = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precisión', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Reg. Logística': [acc_lr, prec_lr, rec_lr, f1_lr, roc_lr],
    'XGBoost': [acc_xgb, prec_xgb, rec_xgb, f1_xgb, roc_xgb]
})

# Agregar diferencia
comparativa_base['Diferencia'] = comparativa_base['XGBoost'] - comparativa_base['Reg. Logística']

print(comparativa_base.to_string(index=False))

# Mejor modelo
mejor_base = 'XGBoost' if roc_xgb > roc_lr else 'Reg. Logística'
print(f"\n Mejor modelo (ROC-AUC): {mejor_base}")
print("=" * 80)

---

## Optimización de Hiperparámetros con RandomizedSearchCV

Se optimizan ambos modelos buscando los mejores hiperparámetros usando RandomizedSearchCV con validación cruzada.


In [ ]:
# RandomizedSearchCV - Regresión Logística
print("\nOPTIMIZACIÓN: REGRESIÓN LOGÍSTICA")
print("=" * 70)

param_dist_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [500, 1000, 1500]
}

random_search_lr = RandomizedSearchCV(
    LogisticRegression(),
    param_dist_lr,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    random_state=RANDOM_STATE_CORRIDA,
    n_jobs=-1,
    verbose=0
)

random_search_lr.fit(X_train_scaled, y_train)

print(f"Mejores parámetros: {random_search_lr.best_params_}")
print(f"ROC-AUC (CV): {random_search_lr.best_score_:.4f}")

# Evaluar en prueba
lr_best = random_search_lr.best_estimator_
y_pred_lr_opt = lr_best.predict(X_test_scaled)
y_proba_lr_opt = lr_best.predict_proba(X_test_scaled)[:, 1]

acc_lr_opt = accuracy_score(y_test, y_pred_lr_opt)
prec_lr_opt = precision_score(y_test, y_pred_lr_opt)
rec_lr_opt = recall_score(y_test, y_pred_lr_opt)
f1_lr_opt = f1_score(y_test, y_pred_lr_opt)
roc_lr_opt = roc_auc_score(y_test, y_proba_lr_opt)

print(f"\nDesempeño en PRUEBA:")
print(f"Accuracy:  {acc_lr_opt:.4f}")
print(f"Precisión: {prec_lr_opt:.4f}")
print(f"Recall:    {rec_lr_opt:.4f}")
print(f"F1-Score:  {f1_lr_opt:.4f}")
print(f"ROC-AUC:   {roc_lr_opt:.4f}")
print("=" * 70)

In [ ]:
# RandomizedSearchCV - XGBoost
print("\nOPTIMIZACIÓN: XGBOOST")
print("=" * 70)

param_dist_xgb = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9]
}

random_search_xgb = RandomizedSearchCV(
    XGBClassifier(random_state=RANDOM_STATE_CORRIDA, eval_metric='logloss', verbose=0),
    param_dist_xgb,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    random_state=RANDOM_STATE_CORRIDA,
    n_jobs=-1,
    verbose=0
)

random_search_xgb.fit(X_train, y_train)

print(f"Mejores parámetros: {random_search_xgb.best_params_}")
print(f"ROC-AUC (CV): {random_search_xgb.best_score_:.4f}")

# Evaluar en prueba
xgb_best = random_search_xgb.best_estimator_
y_pred_xgb_opt = xgb_best.predict(X_test)
y_proba_xgb_opt = xgb_best.predict_proba(X_test)[:, 1]

acc_xgb_opt = accuracy_score(y_test, y_pred_xgb_opt)
prec_xgb_opt = precision_score(y_test, y_pred_xgb_opt)
rec_xgb_opt = recall_score(y_test, y_pred_xgb_opt)
f1_xgb_opt = f1_score(y_test, y_pred_xgb_opt)
roc_xgb_opt = roc_auc_score(y_test, y_proba_xgb_opt)

print(f"\nDesempeño en PRUEBA:")
print(f"Accuracy:  {acc_xgb_opt:.4f}")
print(f"Precisión: {prec_xgb_opt:.4f}")
print(f"Recall:    {rec_xgb_opt:.4f}")
print(f"F1-Score:  {f1_xgb_opt:.4f}")
print(f"ROC-AUC:   {roc_xgb_opt:.4f}")
print("=" * 70)

In [ ]:
# Comparación BASE vs OPTIMIZADO
print("\nCOMPARACIÓN: BASE vs OPTIMIZADO")
print("=" * 80)

# Tabla comparativa completa
comparativa_final = pd.DataFrame({
    'Modelo': ['RL Base', 'RL Optimizado', 'XGB Base', 'XGB Optimizado'],
    'Accuracy': [acc_lr, acc_lr_opt, acc_xgb, acc_xgb_opt],
    'Precisión': [prec_lr, prec_lr_opt, prec_xgb, prec_xgb_opt],
    'Recall': [rec_lr, rec_lr_opt, rec_xgb, rec_xgb_opt],
    'F1-Score': [f1_lr, f1_lr_opt, f1_xgb, f1_xgb_opt],
    'ROC-AUC': [roc_lr, roc_lr_opt, roc_xgb, roc_xgb_opt]
})

print(comparativa_final.to_string(index=False))

# Mejoras
print(f"\n MEJORAS CON OPTIMIZACIÓN:")
print(f"Reg. Logística: {roc_lr:.4f} → {roc_lr_opt:.4f} ({(roc_lr_opt-roc_lr)*100:+.2f}%)")
print(f"XGBoost:        {roc_xgb:.4f} → {roc_xgb_opt:.4f} ({(roc_xgb_opt-roc_xgb)*100:+.2f}%)")

# Mejor modelo final
mejor_final = 'XGBoost Optimizado' if roc_xgb_opt > roc_lr_opt else 'Reg. Logística Optimizada'
print(f"\n MEJOR MODELO FINAL: {mejor_final}")
print(f"   ROC-AUC: {max(roc_lr_opt, roc_xgb_opt):.4f}")
print("=" * 80)